# Bad Controls - Covariates That Treatment Can Affect

A **bad control** is a covariate that treatment can change. Occupation, hours, health
insurance, firm size: anything measured after treatment starts and plausibly moved by
it. Caetano, Callaway, Payne and Sant'Anna (2026) formalize when such a covariate is
"bad" - it is *relevant* for the untreated outcome path (Condition 1) and treatment
*affects* it (Condition 2) - and show that the two conventions practitioners reach for
are both wrong in general:

- **conditioning on the bad control at $t$** compares treated and untreated units at
  different values of $X_t(0)$, so the estimate absorbs part of the treatment's own
  effect on $X$ (their Section 3.1 `tau^use` bias);
- **dropping the bad control** ignores that its untreated path carries information
  about the outcome trend (their Section 3.2 `tau^discard` bias); the paper's Table
  S1 marks the "TWFE, exclude BC" arm inconsistent in every one of its designs. This
  notebook states that result and does not run a "drop X" arm - it is out of scope
  here, not harmless.

The paper offers two identification routes, and both are available here:

1. **Approach 1 - condition on the pre-treatment value.** The maintained
   assumption is parallel trends conditional on the bad control's *untreated* path
   (Assumption 2 / MP-4); Approach 1 adds either *simple covariate unconfoundedness*
   of that path given $(X_{g-1}, Z)$ (Assumption 4 / MP-8) or *bad-control
   redundancy* (Assumption 5 / MP-9), and the conclusion is that conditioning on the
   pre-treatment value $X_{g-1}$ alone identifies the ATT (Theorem 1 / Proposition
   1; staggered: Proposition 3 under MP-1 to MP-4, MP-6, MP-7 and MP-8 or MP-9).
   `CallawaySantAnna` and `DMLDiD` already read `covariates` at each cell's base
   period on the panel lane, so passing the bad control there IS Approach 1.
2. **Covariate unconfoundedness for the bad control's path (Approach 2).** The same
   maintained parallel trends (MP-4), but the bad control's untreated evolution is
   now identified given its pre-treatment value, extra pre-treatment confounders
   $W$ and the clean covariates $Z$ (Assumption 6 / MP-5), again with MP-7 so that
   the untreated outcome change depends on the bad control's history only through
   its two endpoints, $X_{\text{base}}(0)$ and $X_t(0)$ (Proposition 2). Both
   approaches allow treatment to affect $X$; what distinguishes this one is the
   $W$-conditional path-unconfoundedness strategy and the nested doubly-robust
   score that implements it. This is the `DMLDiD` bad-control lane:
   `fit(..., bad_control=, bad_control_covariates=)`.
   It also reports the paper's Remark 6 pre-test $ATT_X(g, t)$ - the effect of
   treatment on the covariate itself - for every cell.

This tutorial:

1. Builds a staggered version of the paper's first Monte Carlo design (DGP 1).
2. Runs the naive TWFE regression with the bad control at $t$ and watches it miss by
   the full treatment effect on $X$.
3. Runs Approach 1 with and without $W$.
4. Runs the bad-control lane and reads its `summary()`.
5. Compares the choices of $W$ (the paper's Remark 5 lagged outcome; none).
6. Reads `bad_control_summary()` - pre-period rows are a pre-test, post-period rows
   are the Condition-2 check - and the event study.
7. Refits with a different learner (the nested stage switches to split-half) and lists
   what the lane will not do.

In [1]:
import warnings

import numpy as np
import pandas as pd

from diff_diff import CallawaySantAnna, DMLDiD, LinearRegression, practitioner_next_steps
from diff_diff.utils import within_transform

pd.set_option("display.precision", 4)

FIT_KW = dict(outcome="y", unit="unit", time="time", first_treat="first_treat")

## 1. The Paper's DGP 1, Staggered

The Supplementary Appendix's first design (SA pp. 14-15) has two periods; we keep every
coefficient and extend it to four periods with two treated cohorts ($g = 3$ and
$g = 4$) plus a never-treated pool. Per unit, with $Z_i, \eta_i \sim N(0, 1)$ and all
noise terms $\varepsilon \sim N(0, 1)$ mutually independent:

$$
\begin{aligned}
W_i &= 0.8\,\eta_i + 0.3\,Z_i + 0.2\,\varepsilon^W_i, &
D_i &= \mathbf{1}\{0.2\,Z_i + 0.4\,W_i + 0.3\,\eta_i + \varepsilon^D_i > 0\}, \\
X_{i1}(0) &= 0.5\,\eta_i + 0.4\,Z_i + 0.3\,\varepsilon^{X}_{i1}, &
X_{it}(0) &= 0.7\,X_{i,t-1}(0) + 0.3\,Z_i + 0.2\,W_i + 0.15 + 0.3\,\varepsilon^{X}_{it}, \\
X_{it}(g) &= X_{it}(0) + \lambda\ \text{for } t \ge g, &
Y_{it}(0) &= 0.3\,t + 0.5\,\eta_i + 0.3\,Z_i + X_{it}(0) + 0.3\,\varepsilon^{Y}_{it}, \\
Y_{it}(g) &= Y_{it}(0) + \lambda + \delta\ \text{for } t \ge g, & \lambda &= \delta = 0.5 .
\end{aligned}
$$

Treated units draw their cohort by a fair coin. The unobserved $\eta_i$ drives
treatment, $W_i$ and the bad control's level, so the bad control's *path* carries the
confounding; $Z_i$ is a clean covariate; $W_i$ is the observed pre-treatment confounder
of $X$. Truth in every post-treatment cell: $ATT(g, t) = \lambda + \delta =
\mathbf{1.00}$ and $ATT_X(g, t) = \lambda = \mathbf{0.50}$; both are $0$ before
treatment.

A note on the seed. The structural assumptions - parallel trends given the bad
control's untreated path, no anticipation, and covariate unconfoundedness of that path
given $(X_{t-1}, W, Z)$ - hold by construction, so pre-period placebo estimates are
pure sampling noise, and a pre-test rejects on roughly 5% of draws even then. One
assumption does *not* hold uniformly: MP-6 requires the probability of staying
untreated given the covariate history to be bounded away from zero - equivalently,
the treatment propensity bounded away from one - and with a Gaussian latent index and
unbounded covariates no such bound exists; this design shares that feature with the
paper's own simulations. The lane's propensity trimming is numerical regularization
of the estimated scores, not a repair of MP-6 - the fits below report how often it
binds. We use `default_rng(5)`, a draw whose pre-period placebos all sit
within one standard error of zero; the first seed we tried produced a 3-SE pre-period
pseudo-ATT in one cell. Read pre-tests jointly across cells and against the size of
the post-treatment effects, never one cell at a time.

In [2]:
n, T = 2000, 4
rng = np.random.default_rng(5)
Z = rng.standard_normal(n)
eta = rng.standard_normal(n)  # unobserved heterogeneity
W = 0.8 * eta + 0.3 * Z + 0.2 * rng.standard_normal(n)
D = (0.2 * Z + 0.4 * W + 0.3 * eta + rng.standard_normal(n)) > 0  # ever treated
g = np.where(D, rng.choice([3, 4], size=n), 0)  # cohort (0 = never treated)
X0 = np.empty((n, T + 1))  # column t = X_t(0), 1-based
X0[:, 1] = 0.5 * eta + 0.4 * Z + 0.3 * rng.standard_normal(n)
for t in range(2, T + 1):
    X0[:, t] = 0.7 * X0[:, t - 1] + 0.3 * Z + 0.2 * W + 0.15 + 0.3 * rng.standard_normal(n)  # DGP 1
frames = []
for t in range(1, T + 1):  # one outcome-noise draw per period, in period order
    post = (g > 0) & (t >= g)
    x_t = X0[:, t] + 0.5 * post  # X_t(g) = X_t(0) + lambda
    y_t = 0.3 * t + 0.5 * eta + 0.3 * Z + X0[:, t] + 0.3 * rng.standard_normal(n) + post * (0.5 + 0.5)
    frames.append(
        pd.DataFrame({"unit": np.arange(n), "time": t, "first_treat": g, "y": y_t, "x": x_t, "z": Z, "w": W})
    )
df = pd.concat(frames, ignore_index=True)
df["post"] = ((df["first_treat"] > 0) & (df["time"] >= df["first_treat"])).astype(float)

df.groupby("unit")["first_treat"].first().value_counts().sort_index()

first_treat
0    974
3    531
4    495
Name: count, dtype: int64

**974** never-treated units against cohorts of **531** ($g = 3$) and **495** ($g = 4$).

## 2. The Naive Regression

The paper's first application estimator is a two-way fixed-effects regression of
$Y_{it}$ on the staggered treatment indicator $D_{it}$ *and the bad control* $X_{it}$
("TWFE: include BC"). We run it as a within regression - unit and period means removed
with `within_transform`, then `LinearRegression` on the demeaned columns with unit
clusters. (The library's `TwoWayFixedEffects` is a $2 \times 2$ estimator: with a
unit-specific `post` indicator its `treatment x post` column coincides with `post`
itself and the fit is rejected as collinear, which is why the two-line within
regression is used here.) The clustered standard error uses the library's absorbed
fixed-effects convention: `cluster_k_adjustment=4` counts the absorbed constant plus
the $T - 1 = 3$ period effects that are not nested in the unit clusters
(`docs/methodology/variance-conventions.md`, defect D2).

In [3]:
def twfe(cols):
    # Within regression of y on [D_it, *cols] with unit and period effects removed.
    d = within_transform(df, variables=["y", "post", *cols], unit="unit", time="time")
    X = d[[f"{c}_demeaned" for c in ["post", *cols]]].to_numpy()
    y = d["y_demeaned"].to_numpy()
    lr = LinearRegression(include_intercept=False, cluster_ids=df["unit"].to_numpy()).fit(
        X, y, cluster_k_adjustment=4
    )
    return float(lr.coefficients_[0]), float(np.sqrt(lr.vcov_[0, 0]))


rows = []
att, se = twfe(["x"])
rows.append(dict(estimator="TWFE, include X_t", att=att, se=se, bias=att - 1.0))
pd.DataFrame(rows)

,estimator,att,se,bias
0,"TWFE, include X_t",0.4886,0.0142,-0.5114


**0.4886 ± 0.0142** against a true **1.00**: the regression nets out the part of the
treatment effect that runs through the covariate ($\lambda = 0.5$), matching the
paper's Table S2 bias of $-0.501$ at $n = 2000$. Nothing about the sample size fixes
this.

## 3. Approach 1 - Condition on the Pre-Treatment Value

On the panel lane, `CallawaySantAnna` reads every covariate at the cell's base period
(the period before the cohort's first treatment for post-treatment cells), never at
$t$. Passing the bad control in `covariates` therefore conditions on $X_{g-1}$ - the
paper's Theorem 1 / Proposition 3 estimand. The estimator reads the supplied column at
the base row, but it cannot tell whether that value contains only information
available then: a lead, a copied post-treatment value or any other future-informed
column would silently recreate the Section 2 bias (the paper's Section 3.1 `tau^use` bias), so supplying a genuine
pre-treatment value is the user's responsibility. We use the not-yet-treated comparison group throughout,
the group the paper's staggered results are stated for.

The assumption map for Approach 1 is worth stating precisely, because none of its
labels is a "parallel trends given $X_{g-1}$" assumption. Two periods (Theorem 1 and
Proposition 1): Assumptions 1-3 (absorbing treatment, no anticipation, parallel
trends conditional on the bad control's untreated path and $Z$) plus *either*
Assumption 4, simple covariate unconfoundedness $X_{t^*}(0) \perp D \mid X_{t^*-1},
Z$, *or* Assumption 5, bad-control redundancy. Staggered (Proposition 3): MP-1 to
MP-4, overlap MP-6, the endpoint-only reduction MP-7 - the untreated outcome change
may depend on the bad control's history only through $X_{g-1}(0)$ and $X_t(0)$, which
holds here because $Y_t(0) - Y_b(0) = 0.3(t - b) + X_t(0) - X_b(0) + \varepsilon^Y_t
- \varepsilon^Y_b$ - plus MP-8 or MP-9. What these deliver is that conditioning on
$X_{g-1}$ and $Z$ *alone* identifies the ATT. In this design that is
not enough: the bad control's untreated evolution depends on $W$, which is correlated
with treatment through $\eta$, so the paper's own "ML (Pre-treatment)" arm is
inconsistent here (Table S1; bias $+0.042$ at $n = 2000$). Adding $W$ to the
conditioning set treats $(W, Z)$ as the expanded Approach-1 covariate set: MP-4 and
MP-8 must then hold relative to that set, they do here ($X_t(0)$ given $X_{t-1}, W, Z$
no longer depends on $\eta$), and the estimate recovers the truth. In this linear design every route that conditions on $W$ agrees. Both approaches
allow treatment to move $X$; what the bad-control lane adds is Approach 2's
identification strategy - unconfoundedness of the bad control's untreated path given
$(X_{\text{base}}, W, Z)$, implemented by the nested doubly-robust score - and the
$ATT_X$ surface.

In [4]:
with warnings.catch_warnings():
    # (one trimmed propensity per small cell; the same warning is shown and
    #  explained in Section 4)
    warnings.simplefilter("ignore")
    cs_a1 = CallawaySantAnna(control_group="not_yet_treated").fit(df, **FIT_KW, covariates=["x", "z"])
    cs_a1w = CallawaySantAnna(control_group="not_yet_treated").fit(
        df, **FIT_KW, covariates=["x", "z", "w"]
    )
for label, r in (("CS, Approach 1: X_{g-1}, Z", cs_a1), ("CS, Approach 1: X_{g-1}, Z, W", cs_a1w)):
    rows.append(dict(estimator=label, att=r.att, se=r.se, bias=r.att - 1.0))
pd.DataFrame(rows)

,estimator,att,se,bias
0,"TWFE, include X_t",0.4886,0.0142,-0.5114
1,"CS, Approach 1: X_{g-1}, Z",1.0540,0.0235,0.0540
2,"CS, Approach 1: X_{g-1}, Z, W",0.9754,0.0274,-0.0246


Approach 1 as the paper states it lands at **1.0540 ± 0.0235** (biased by two
standard errors); with $W$ in the conditioning set it lands at **0.9754 ± 0.0274**,
within one standard error of the truth.

## 4. The Bad-Control Lane

`DMLDiD` keeps its Callaway-Sant'Anna cell architecture and swaps the per-cell score
for the paper's Neyman-orthogonal doubly-robust score (their Equation 10) when a
`bad_control` is declared. The per-cell estimand is Proposition 2's: parallel trends
given the bad control's untreated path (MP-4), covariate unconfoundedness of that
path (MP-5), overlap (MP-6) and the endpoint-only reduction MP-7 that lets the
score condition on $(X_{\text{base}}, X_t)$ instead of the full history. Four
nuisances are cross-fitted per cell:

- the control outcome-change regression $m_0$ on $R = (X_t, X_{\text{base}}, Z)$,
- the propensity $p$ on $S = (X_{\text{base}}, W, Z)$,
- and two *nested* second stages fit on the training controls only: $\nu_0$, the
  first-stage predictions regressed on $S$, and $\omega_0$, the propensity odds
  regressed on $R$.

The base period is $g - 1$ for post-treatment cells and the immediately preceding
period for the pre-treatment pseudo-cells (the library's varying-base convention).
`bad_control_covariates` is the paper's $W$; the outcome name is allowed there and
means the outcome at the base period (Remark 5). `covariates` stays required and is
the paper's $Z$; the bad control must not appear in `covariates`.

Two kinds of warning are raised by this fit and are worth reading rather than
silencing; the cell below collects them and prints their messages. The
$\omega_0$ stage projects propensity *odds* with a linear regression, which can go
negative; the lane clips $\hat\omega$ to $[0, (1 - \text{trim}) / \text{trim}]$ and
warns with the count per cell (the paper gives no rule; the choice is recorded as a
REGISTRY Note). The three cells with fewer than 2,000 units also trim one fitted
propensity to the `pscore_trim` bounds - the existing propensity diagnostic, which
is raised once per affected cell.

In [5]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    res = DMLDiD(control_group="not_yet_treated", seed=0).fit(
        df, **FIT_KW, covariates=["z"], bad_control="x", bad_control_covariates=["w"]
    )
print(res.summary())
print(
    "n_clipped_omega per cell:",
    {(int(g_), int(t_)): d["n_clipped_omega"] for (g_, t_), d in res.cross_fit_diagnostics.items()},
)
print(f"\n{len(caught)} warnings raised by the fit:")
for w in caught:
    print(f"  {w.category.__name__}: {w.message}")
rows.append(dict(estimator="DMLDiD bad-control lane, W = [w]", att=res.att, se=res.se, bias=res.att - 1.0))

  DML DiD (CCPS 2026 bad-control score) Staggered Difference-in-Differences Results  

Propensity learner:               'logit'
Outcome learner:                 'linear'
Cross-fitting folds (K):                5
Seed:                                   0
Bad control:                            x
Bad-control covariates (W):             w
ATT_X cells (post / pre):           3 / 3
Overall ATT weighting: CS simple (not Remark 4; REGISTRY DMLDiD Note).
Call results.bad_control_summary() for the ATT_X pre-test table.

Total observations:                  8000
Treated units:                       1026
Never-treated units:                  974
Treatment cohorts:                      2
Time periods:                           4
Control group:                 not_yet_treated
Base period:                      varying

-------------------------------------------------------------------------------------
                   Overall Average Treatment Effect on the Treated                   
----------

The header records the lane (`CCPS 2026 bad-control score`), the bad control, $W$,
how many $ATT_X$ cells are post- and pre-treatment, and that the headline ATT uses the
Callaway-Sant'Anna simple weighting rather than the paper's Remark 4 overall. The
overall estimate is **0.9766 ± 0.0260**, within one standard error of the truth.

## 5. Choosing $W$

$W$ is whatever pre-treatment information makes the bad control's untreated evolution
unconfounded (MP-5). The paper's Remark 5 discusses the base-period outcome, $Y_{g-1}$,
as the natural candidate when covariate unconfoundedness is credible *given the lagged
outcome*; passing the outcome name in `bad_control_covariates` does exactly that. In
this design it is a deliberately misspecified sensitivity fit: $X_t(0)$ still depends
on the true $W$, treatment depends on $W$ and $\eta$, and $Y_{g-1}$ is only a noisy
proxy for both, so MP-5 does **not** hold with `W=["y"]` here. The estimate moves to
**1.0169 ± 0.0259**; that it stays within one standard error of the truth is a
property of this draw, not an identification diagnostic. Declaring no $W$ at all
(`bad_control_covariates=None`, the R package's default) is the analogue of Approach 1
without $W$ and lands at **1.0591 ± 0.0235**.

In [6]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # (same omega-clip / trim warnings as above)
    res_y = DMLDiD(control_group="not_yet_treated", seed=0).fit(
        df, **FIT_KW, covariates=["z"], bad_control="x", bad_control_covariates=["y"]
    )
    res_now = DMLDiD(control_group="not_yet_treated", seed=0).fit(
        df, **FIT_KW, covariates=["z"], bad_control="x"
    )
rows.append(dict(estimator="DMLDiD bad-control lane, W = [y] (Remark 5)", att=res_y.att, se=res_y.se, bias=res_y.att - 1.0))
rows.append(dict(estimator="DMLDiD bad-control lane, no W", att=res_now.att, se=res_now.se, bias=res_now.att - 1.0))
pd.DataFrame(rows)

,estimator,att,se,bias
0,"TWFE, include X_t",0.4886,0.0142,-0.5114
1,"CS, Approach 1: X_{g-1}, Z",1.0540,0.0235,0.0540
2,"CS, Approach 1: X_{g-1}, Z, W",0.9754,0.0274,-0.0246
3,"DMLDiD bad-control lane, W = [w]",0.9766,0.0260,-0.0234
4,"DMLDiD bad-control lane, W = [y] (Remark 5)",1.0169,0.0259,0.0169
5,"DMLDiD bad-control lane, no W",1.0591,0.0235,0.0591


## 6. Reading `bad_control_summary()`

Every retained cell carries $\widehat{ATT}_X(g, t)$: an AIPW estimate of the effect of
treatment on the bad control itself, with its own analytical standard error. The
reading depends on the period:

- **Pre-treatment rows** ($t < g$) are the paper's Remark 6 pre-test of the
  identifying assumptions for the bad control's untreated path (MP-5 / MP-8). They
  should be zero. A nonzero pre-period value flags a possible violation of those
  assumptions - it is *not* evidence that the covariate is a bad control.
- **Post-treatment rows** ($t \ge g$) are the Condition-2 check. A nonzero value is
  evidence that treatment moves the covariate; a zero value does not establish the
  converse (heterogeneous effects can cancel in the mean), and $ATT_X$ says nothing
  about Condition 1.

Here the three pre-period rows are **0.0019**, **0.0120** and **0.0061** (standard
errors 0.017-0.019) and the three post-period rows are **0.5137**, **0.5055** and
**0.4764** against the true $\lambda = 0.50$. Remember the seed note from Section 1:
with the assumptions holding by construction, a pre-period row a few standard errors
from zero is a draw, not a diagnosis, on roughly one seed in twenty (the structural
assumptions hold by construction; only uniform overlap does not, see Section 1).

The ATT itself aggregates exactly as on the plain lane. The event study below shows the
analytical confidence intervals; the simultaneous `cband_*` columns need
`n_bootstrap > 0` and are omitted.

In [7]:
print(res.bad_control_summary().to_string(index=False))
res.aggregate("event_study").to_dataframe()[["event_time", "att", "se", "conf_int_lower", "conf_int_upper"]]

 group  time  post  att_x   se_x  t_stat     p_value  conf_int_lower  conf_int_upper
     3     2 False 0.0019 0.0166  0.1132  9.0990e-01         -0.0307          0.0345
     3     3  True 0.5137 0.0156 32.9304 8.0768e-238          0.4831          0.5442
     3     4  True 0.5055 0.0299 16.9082  3.9136e-64          0.4469          0.5641
     4     2 False 0.0120 0.0169  0.7131  4.7575e-01         -0.0210          0.0451
     4     3 False 0.0061 0.0190  0.3211  7.4810e-01         -0.0311          0.0433
     4     4  True 0.4764 0.0263 18.1414  1.5012e-73          0.4250          0.5279


,event_time,att,se,conf_int_lower,conf_int_upper
0,-2,0.0128,0.0281,-0.0423,0.0680
1,-1,-0.0044,0.0208,-0.0450,0.0363
2,0,0.9837,0.0226,0.9393,1.0281
3,1,0.9629,0.0412,0.8822,1.0437


Event times $-2$ and $-1$ sit at **0.0128** and **-0.0044**; event times $0$ and $1$ at
**0.9837** and **0.9629**.

## 7. Learner Sensitivity, and What the Lane Will Not Do

Baker et al.'s step 8 is a refit under a different specification. With the parametric
built-in learners the nested second stages use the first stage's in-sample fitted
values as their targets - the paper's own plug-in for its linear working models. With
any other learner (`ridge`, `sieve`, or a user object) the lane instead splits each
training fold in half, fits the first stage on one half and the nested stage on the
other, swaps, and averages (the paper's footnote 9); `cross_fit_diagnostics` records
which branch ran as `nested_stage`.

Restrictions worth knowing before reaching for the lane:

- **Panel data only** (the paper's Remark 1): `panel=False` with a bad control raises.
- **`cluster=` yes, `survey_design=` no**; `anticipation=0` and the varying base
  period only. Each unsupported combination raises `NotImplementedError` rather than
  estimating something else.
- **The bad control must not appear in `covariates`** (the fit raises) - that is the
  Section 2 bias (the paper's Section 3.1 `tau^use` bias). To run Approach 1 instead, drop `bad_control` and pass the column in
  `covariates` alone.
- **$ATT_X$ is analytical only**: never bootstrapped, never aggregated.
- **The headline ATT keeps the Callaway-Sant'Anna simple weighting**, not the paper's
  Remark 4 cohort-mass weighting; `summary()` says so.
- The lane is validated as a black box against the authors' R package `badcontrols`
  (tolerance-based, because the R package cross-fits with its own fold draw) and
  against the paper's Monte Carlo designs; the REGISTRY entry lists every convention.

In [8]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # (same omega-clip / trim warnings as above)
    res_ridge = DMLDiD(control_group="not_yet_treated", seed=0, outcome_learner="ridge").fit(
        df, **FIT_KW, covariates=["z"], bad_control="x", bad_control_covariates=["w"]
    )
print("nested_stage:", {d["nested_stage"] for d in res_ridge.cross_fit_diagnostics.values()})
rows.append(dict(estimator="DMLDiD bad-control lane, W = [w], ridge", att=res_ridge.att, se=res_ridge.se, bias=res_ridge.att - 1.0))
pd.DataFrame(rows)

nested_stage: {'split_half'}


,estimator,att,se,bias
0,"TWFE, include X_t",0.4886,0.0142,-0.5114
1,"CS, Approach 1: X_{g-1}, Z",1.0540,0.0235,0.0540
2,"CS, Approach 1: X_{g-1}, Z, W",0.9754,0.0274,-0.0246
3,"DMLDiD bad-control lane, W = [w]",0.9766,0.0260,-0.0234
4,"DMLDiD bad-control lane, W = [y] (Remark 5)",1.0169,0.0259,0.0169
5,"DMLDiD bad-control lane, no W",1.0591,0.0235,0.0591
6,"DMLDiD bad-control lane, W = [w], ridge",0.9770,0.0261,-0.0230


The ridge refit lands at **0.9770 ± 0.0261** through the split-half branch: the
estimate is insensitive to this linear-to-ridge refit on this draw. (A materially
different learner class, and a refit of the propensity learner, would be needed
before claiming more.)

## Summary

| Situation | Reach for |
|-----------|-----------|
| A covariate that treatment can change, parallel trends given its untreated path (Assumption 2 / MP-4), and either simple covariate unconfoundedness (Assumption 4 / MP-8) or redundancy (Assumption 5 / MP-9) | `CallawaySantAnna` or `DMLDiD` with the column in `covariates` (read at the base period: Approach 1) |
| The same covariate, identifying its untreated path through covariate unconfoundedness given its pre-treatment value, named confounders $W$ and $Z$ (Assumption 6 / MP-5; Approach 2) | `DMLDiD(...).fit(..., bad_control=, bad_control_covariates=[...])` |
| You want to know whether treatment moved the covariate at all, or pre-test the path assumptions | `results.bad_control_summary()` (post rows / pre rows) |
| Conditioning on the covariate at $t$ | Never - that is the Section 2 bias (the paper's Section 3.1 `tau^use` bias). The panel API reads `covariates` at the base row (a future-informed column is not detectable, so keep it genuinely pre-treatment), and the bad-control lane rejects a column that is both `bad_control` and in `covariates` |
| Repeated cross-sections, survey designs, anticipation, universal base period | Not available on the bad-control lane (fails closed); see the tracked rows |

**References**: Caetano, C., Callaway, B., Payne, S., & Sant'Anna, H. (2026).
Difference-in-differences with "bad controls". arXiv:2608.03881. | Chang, N.-C.
(2020). Double/debiased machine learning for difference-in-differences models. *The
Econometrics Journal*, 23(2), 177-191.

## What next

Fitted results know their own follow-up work: `practitioner_next_steps()` returns the
Baker, Callaway, Cunningham, Goodman-Bacon & Sant'Anna (2025) practitioner workflow
steps as runnable templates. On a bad-control fit the learner-sensitivity template
carries `bad_control=`, `bad_control_covariates=` and the comparison group forward, so
the refit targets the same estimand.

In [9]:
guidance = practitioner_next_steps(res)


Practitioner Guidance — DMLDiD (CCPS 2026 bad-control score)
Baker et al. (2025) 8-Step Workflow

Recommended next steps (7 remaining):

  * [HIGH] Step 1: Define target parameter
    Why: State explicitly what causal effect you are estimating (ATT, ATT(g,t), weighted/unweighted) and what policy question it answers.
    >>> # What is the target parameter? ATT? Weighted or unweighted?

  * [HIGH] Step 2: State identification assumptions
    Why: Name the parallel trends variant you are invoking (unconditional, conditional, PT-GT-NYT, etc.), the no-anticipation assumption, and any overlap conditions.
    >>> # Which PT variant? No-anticipation? Overlap?

  * [HIGH] Step 3: Test parallel trends (event-study pre-periods)
    Why: For staggered designs, inspect event-study pre-period coefficients rather than the generic check_parallel_trends() which assumes a single binary treatment with universal pre-periods. Pre-treatment ATTs should be near zero. Use CS post-fit results.aggregate('event